# Baseline Models

## Objective
Establish baseline forecasting performance using simple models:
1. **Naive**: Tomorrow = Today
2. **Seasonal Naive**: Tomorrow = Same day last week
3. **Moving Average**: Tomorrow = Average of last N days
4. **Linear Regression**: With engineered features

## Importance of Baselines
- Provide performance benchmarks for complex models
- Often surprisingly competitive
- Help validate that complex models actually add value
- Simple to interpret and explain

In [1]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import project modules
import src.config as config
from src.utils import print_section_header
from src.models import (
    NaiveForecaster,
    SeasonalNaiveForecaster,
    MovingAverageForecaster,
    LinearRegressionForecaster
)

# Set random seed
np.random.seed(config.RANDOM_SEED)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')

print("Environment setup complete!")

Environment setup complete!


## 1. Load Preprocessed Data

In [2]:
# Load data
X_train = np.load(config.DATA_PATH / 'X_train.npy')
y_train = np.load(config.DATA_PATH / 'y_train.npy')
X_test = np.load(config.DATA_PATH / 'X_test.npy')
y_test = np.load(config.DATA_PATH / 'y_test.npy')
train_dates = np.load(config.DATA_PATH / 'train_dates.npy', allow_pickle=True)
test_dates = np.load(config.DATA_PATH / 'test_dates.npy', allow_pickle=True)

# Load feature names
with open(config.DATA_PATH / 'feature_names.txt', 'r') as f:
    feature_names = [line.strip() for line in f]

print(f"Data loaded successfully!")
print(f"Train: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test: {X_test.shape[0]} samples, {X_test.shape[1]} features")

Data loaded successfully!
Train: 1404 samples, 21 features
Test: 254 samples, 21 features


## 2. Define Evaluation Metrics

We'll use:
- **MAE (Mean Absolute Error)**: Average absolute difference between predictions and actuals
- **RMSE (Root Mean Squared Error)**: Square root of average squared errors (penalizes large errors more)

In [3]:
def compute_metrics(y_true, y_pred):
    """
    Compute MAE and RMSE.
    
    Parameters:
    -----------
    y_true : array-like
        True values
    y_pred : array-like
        Predicted values
    
    Returns:
    --------
    dict
        Dictionary with MAE and RMSE
    """
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    
    return {'MAE': mae, 'RMSE': rmse}

def print_metrics(model_name, metrics):
    """Print metrics in a formatted way."""
    print(f"\n{model_name}")
    print("-" * 50)
    print(f"  MAE:  {metrics['MAE']:.2f}")
    print(f"  RMSE: {metrics['RMSE']:.2f}")

## 3. Baseline Model 1: Naive Forecaster

**Method**: Predict tomorrow's sales = today's sales  
**Formula**: ŷ(t) = y(t-1)

This is the simplest possible forecasting method.

In [4]:
# Initialize and fit Naive model
naive_model = NaiveForecaster()
naive_model.fit(y_train)

# Make predictions
naive_pred = naive_model.predict(steps=len(y_test))

# Evaluate
naive_metrics = compute_metrics(y_test, naive_pred)
print_metrics("Naive Forecaster", naive_metrics)


Naive Forecaster
--------------------------------------------------
  MAE:  8565.22
  RMSE: 9220.62


## 4. Baseline Model 2: Seasonal Naive Forecaster

**Method**: Predict tomorrow's sales = same day last week's sales  
**Formula**: ŷ(t) = y(t-7)  
**Rationale**: Captures weekly seasonality (Monday sales similar to last Monday, etc.)

In [5]:
# Initialize and fit Seasonal Naive model
seasonal_naive_model = SeasonalNaiveForecaster(seasonal_period=7)
seasonal_naive_model.fit(y_train)

# Make predictions
seasonal_naive_pred = seasonal_naive_model.predict(steps=len(y_test))

# Evaluate
seasonal_naive_metrics = compute_metrics(y_test, seasonal_naive_pred)
print_metrics("Seasonal Naive (7-day)", seasonal_naive_metrics)


Seasonal Naive (7-day)
--------------------------------------------------
  MAE:  8466.06
  RMSE: 9175.24


## 5. Baseline Model 3: Moving Average

**Method**: Predict tomorrow's sales = average of last N days  
**Formula**: ŷ(t) = mean(y(t-N), ..., y(t-1))  
**Rationale**: Smooths out noise by averaging recent history

In [6]:
# Initialize and fit Moving Average model
ma_model = MovingAverageForecaster(window=7)
ma_model.fit(y_train)

# Make predictions
ma_pred = ma_model.predict(steps=len(y_test))

# Evaluate
ma_metrics = compute_metrics(y_test, ma_pred)
print_metrics("Moving Average (7-day)", ma_metrics)


Moving Average (7-day)
--------------------------------------------------
  MAE:  3416.86
  RMSE: 3924.05


## 6. Baseline Model 4: Linear Regression

**Method**: Linear regression with engineered features  
**Features**: Lags, rolling stats, calendar features, promotions  
**Rationale**: Simple ML baseline that uses all available features

In [7]:
# Initialize and fit Linear Regression model
lr_model = LinearRegressionForecaster()
lr_model.fit(y_train, X_train)

# Make predictions
lr_pred = lr_model.predict(X_test=X_test)

# Evaluate
lr_metrics = compute_metrics(y_test, lr_pred)
print_metrics("Linear Regression", lr_metrics)


Linear Regression
--------------------------------------------------
  MAE:  1669.27
  RMSE: 2556.15


In [8]:
# Feature importance from Linear Regression
lr_importance = lr_model.get_feature_importance(feature_names=feature_names)
print("\nTop 10 Most Important Features (by absolute coefficient):")
print(lr_importance.head(10))


Top 10 Most Important Features (by absolute coefficient):
           feature  coefficient
15      is_weekend  3687.750775
17    is_month_end   951.283075
16  is_month_start   921.839421
19   has_promotion   448.949739
14         quarter    69.653045
12           month    54.529696
13            year    52.453585
11    day_of_month   -42.052871
18     onpromotion     2.030048
10     day_of_week     1.907601


## 7. Baseline Comparison

In [9]:
# Create comparison table
results = {
    'Naive': naive_metrics,
    'Seasonal Naive (7-day)': seasonal_naive_metrics,
    'Moving Average (7-day)': ma_metrics,
    'Linear Regression': lr_metrics
}

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('MAE')

print_section_header("Baseline Model Comparison")
print(results_df)

# Identify best baseline
best_model = results_df.index[0]
best_mae = results_df.loc[best_model, 'MAE']
print(f"\n✓ Best baseline model: {best_model}")
print(f"  MAE: {best_mae:.2f}")


 Baseline Model Comparison

                                MAE         RMSE
Linear Regression       1669.273479  2556.153372
Moving Average (7-day)  3416.858830  3924.052913
Seasonal Naive (7-day)  8466.059055  9175.241916
Naive                   8565.224409  9220.623933

✓ Best baseline model: Linear Regression
  MAE: 1669.27


In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

results_df.plot(kind='barh', ax=ax, alpha=0.8, edgecolor='black')
ax.set_xlabel('Error', fontsize=12)
ax.set_ylabel('Model', fontsize=12)
ax.set_title('Baseline Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(title='Metric')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()

# Save figure
from src.utils import save_figure
save_figure(fig, '13_baseline_comparison.png')
plt.show()

## 8. Visualize Predictions vs Actuals

In [ ]:
# Create a comprehensive plot of all models
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

test_dates_dt = pd.to_datetime(test_dates)

# Plot 1: Naive
axes[0].plot(test_dates_dt, y_test, label='Actual', linewidth=2, alpha=0.8, color='black')
axes[0].plot(test_dates_dt, naive_pred, label='Predicted', linewidth=1.5, alpha=0.8, color='blue')
axes[0].set_ylabel('Sales', fontsize=11)
axes[0].set_title(f'Naive Forecaster (MAE: {naive_metrics["MAE"]:.2f})', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Plot 2: Seasonal Naive
axes[1].plot(test_dates_dt, y_test, label='Actual', linewidth=2, alpha=0.8, color='black')
axes[1].plot(test_dates_dt, seasonal_naive_pred, label='Predicted', linewidth=1.5, alpha=0.8, color='green')
axes[1].set_ylabel('Sales', fontsize=11)
axes[1].set_title(f'Seasonal Naive (MAE: {seasonal_naive_metrics["MAE"]:.2f})', fontsize=12, fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Plot 3: Moving Average
axes[2].plot(test_dates_dt, y_test, label='Actual', linewidth=2, alpha=0.8, color='black')
axes[2].plot(test_dates_dt, ma_pred, label='Predicted', linewidth=1.5, alpha=0.8, color='orange')
axes[2].set_ylabel('Sales', fontsize=11)
axes[2].set_title(f'Moving Average (MAE: {ma_metrics["MAE"]:.2f})', fontsize=12, fontweight='bold')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

# Plot 4: Linear Regression
axes[3].plot(test_dates_dt, y_test, label='Actual', linewidth=2, alpha=0.8, color='black')
axes[3].plot(test_dates_dt, lr_pred, label='Predicted', linewidth=1.5, alpha=0.8, color='red')
axes[3].set_ylabel('Sales', fontsize=11)
axes[3].set_xlabel('Date', fontsize=11)
axes[3].set_title(f'Linear Regression (MAE: {lr_metrics["MAE"]:.2f})', fontsize=12, fontweight='bold')
axes[3].legend(loc='upper right')
axes[3].grid(True, alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()
save_figure(fig, '14_baseline_predictions.png')
plt.show()

## 9. Error Analysis

In [ ]:
# Calculate errors for each model
naive_errors = y_test - naive_pred
seasonal_naive_errors = y_test - seasonal_naive_pred
ma_errors = y_test - ma_pred
lr_errors = y_test - lr_pred

# Plot error distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Naive
axes[0, 0].hist(naive_errors, bins=30, alpha=0.7, color='blue', edgecolor='black')
axes[0, 0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Error', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Naive - Error Distribution', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Seasonal Naive
axes[0, 1].hist(seasonal_naive_errors, bins=30, alpha=0.7, color='green', edgecolor='black')
axes[0, 1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Error', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Seasonal Naive - Error Distribution', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Moving Average
axes[1, 0].hist(ma_errors, bins=30, alpha=0.7, color='orange', edgecolor='black')
axes[1, 0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Error', fontsize=11)
axes[1, 0].set_ylabel('Frequency', fontsize=11)
axes[1, 0].set_title('Moving Average - Error Distribution', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Linear Regression
axes[1, 1].hist(lr_errors, bins=30, alpha=0.7, color='red', edgecolor='black')
axes[1, 1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Error', fontsize=11)
axes[1, 1].set_ylabel('Frequency', fontsize=11)
axes[1, 1].set_title('Linear Regression - Error Distribution', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
save_figure(fig, '15_error_distributions.png')
plt.show()

## 10. Summary

### Baseline Performance Results:

**Models Tested:**
1. Naive (previous day)
2. Seasonal Naive (same day last week)
3. Moving Average (7-day window)
4. Linear Regression (with features)

### Key Findings:

- **Best Baseline**: [Will be determined by results]
- **Worst Baseline**: [Will be determined by results]
- **Linear Regression** uses all engineered features and likely performs best
- **Seasonal Naive** likely captures weekly patterns better than simple Naive
- Error distributions show [bias/variance patterns to be observed]

### Next Steps:

1. **Advanced Statistical Models**:
   - SARIMA (captures trend + seasonality)
   - SARIMAX (includes exogenous variables like promotions)

2. **Machine Learning Models**:
   - RandomForest (non-linear patterns)
   - GradientBoosting (better performance, hyperparameter tuning)

3. **Target Performance**:
   - Beat best baseline MAE
   - Improve error distribution (reduce large errors)
   - Better capture seasonality and trend

### Baseline MAE to Beat:
Any advanced model must outperform the baseline MAE to be considered valuable.

In [13]:
# Save results for comparison in next notebooks
baseline_results = {
    'model_names': list(results.keys()),
    'predictions': {
        'naive': naive_pred,
        'seasonal_naive': seasonal_naive_pred,
        'moving_average': ma_pred,
        'linear_regression': lr_pred
    },
    'metrics': results
}

# Save to file
import pickle
with open(config.DATA_PATH / 'baseline_results.pkl', 'wb') as f:
    pickle.dump(baseline_results, f)

print("\n✓ Baseline results saved for comparison with advanced models")


✓ Baseline results saved for comparison with advanced models
